## My predictions

**1. Which language will need the most tokens?**

I expect **Chinese to be the one with most tokens**. I expect **English to need the fewest tokens** and **Turkish in between**.

**2. What will the larger BPE vocabulary change?**

Fewer tokens for every language and more characters per token. I expect the **biggest improvement for Chinese** since 10,000 slots are enough to give most common characters their own token. English should improve too and Turkish should sit between them: frequent stems and suffixes become tokens, but the huge number of different inflected forms is still split.

**3. What kinds of units will BPE learn?**

- **English:** frequent whole words (`the`, `of`, `and`, ` in`), common suffixes/prefixes (`ing`, `tion`, `ed`, `re`). Note that a leading space will be attached to word tokens because of how byte-level BPE handles whitespace.
- **Turkish:** I dont know much about the language so im planning to experiment it as i go.
- **Chinese:** Same as Turkish.

**4. Which tokenizer will work best for language modeling?**

My guess is the **BPE-2k or BPE-10k** and I lean to **BPE-2k** on this small dataset : shorter sequences than character-level (so the Transformer sees more context per position and training is cheaper) but with a small embedding table and few rare tokens that appear too seldom to be learned well. The character-level model should have the longest sequences and must learn spelling from scratch, so I expect it to be worst or at least the slowest. I'm least sure about BPE-10k: it has the shortest sequences, but many of its tokens will be rare in such a small corpus.
    

### Part 1

In [1]:
import os, random
import pandas as pd
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

DATA_DIR = "/srv/data/lt2326-h26/a1"

OUT_DIR = "outputs_part1"
os.makedirs(OUT_DIR, exist_ok=True)

LANGS = ["en", "tr", "zh"]           # the three languages
SEED = 2326                          # same seed as the dataset metadata
random.seed(SEED)

## 1.1 Load the data
Each file has one sentence per line. We read **train** to build tokenizers and **valid** to analyse them

In [2]:
def read_lines(path):
    # Reading a text file, one sentence per line, skipping empty lines
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]

train = {lang: read_lines(f"{DATA_DIR}/train/{lang}.txt") for lang in LANGS}
valid = {lang: read_lines(f"{DATA_DIR}/valid/{lang}.txt") for lang in LANGS}

# check: sentence and character counts if it matches metadata.json
for lang in LANGS:
    print(lang,
          "| train sentences:", len(train[lang]), "chars:", sum(len(s) for s in train[lang]),
          "| valid sentences:", len(valid[lang]), "chars:", sum(len(s) for s in valid[lang]))

en | train sentences: 36004 chars: 2946215 | valid sentences: 4540 chars: 368269
tr | train sentences: 30379 chars: 2946140 | valid sentences: 3735 chars: 368383
zh | train sentences: 75962 chars: 2946177 | valid sentences: 9554 chars: 368266


## 1.2 Tokenizer 1: character level

The vocabulary is every distinct character seen in the **training** data only.
A character that first shows up in validation/test (for example: a rare Chinese character) is mapped to a special `<unk>` token instead of crashing.
I also count how often that happens, so we know how much information this loses.

In [3]:
class CharTokenizer:
    def __init__(self, sentences):
        # sorted set of all characters in the training text 
        #id 0 is reserved for <unk>
        chars = sorted(set("".join(sentences)))
        self.itos = ["<unk>"] + chars                        # id ---> token
        self.stoi = {c: i for i, c in enumerate(self.itos)}  # token ----> id

    def encode(self, text):
        # unseen characters get id 0 (<unk>)
        return [self.stoi.get(c, 0) for c in text]

    def encode_batch(self, sentences):
        return [self.encode(s) for s in sentences]

    def to_tokens(self, text):
        # human-readable tokens (for the examples section)
        return [c if c in self.stoi else "<unk>" for c in text]

    def vocab_size(self):
        return len(self.itos)

    def unk_id(self):
        return 0

# Building it from aLL training sentences of all three languages (one shared vocabulary)
all_train = [s for lang in LANGS for s in train[lang]]
char_tok = CharTokenizer(all_train)
print("Character vocabulary size:", char_tok.vocab_size())

Character vocabulary size: 9443


## 1.3 Tokenizers 2 and 3: BPE with a small and a large vocabulary

**Choices and why**
- **Library:** Hugging Face `tokenizers` 
- **Byte-level BPE:** the starting alphabet is the 256 possible bytes, not characters. This has two advantages: (a) *no unknown tokens ever*, any text can be encoded; (b) the base alphabet is only 256 symbols. Chinese alone has thousands of distinct characters, so a normal character based BPE with a 2000 token vocabulary could not even fit all of them. The downside is that a rare Chinese character can be split into multiple byte pieces.
- **Vocabulary sizes: 2000 and 10000.** 2000 is small: 256 bytes + ~1,750 merges, so it stays close to characters for the harder languages. 10000 is 5x bigger, enough for many whole words and common Chinese characters. The gap is big enough for differences to show, while the embedding table remains small for my small Transformer later.
- **Training data:** `balanced.txt`, the shared multilingual training sentences 

In [ ]:
def train_bpe(vocab_size):
    # an empty BPE model
    tok = Tokenizer(models.BPE())
    # normalise text to Unicode
    tok.normalizer = normalizers.NFC()
    # byte level pre-tokenizer: turns text into bytes and first splits it into rough chunks
    #(words / punctuation) so merges do not cross word boundaries
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    #trainer: keep merging the most frequent pair until we reach vocab_size
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<unk>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # all 256 bytes are always in the vocab
        show_progress=False,
    )
    tok.train([f"{DATA_DIR}/tokenizer/balanced.txt"], trainer)
    return tok

bpe_2k  = train_bpe(2000)
bpe_10k = train_bpe(10000)
print("BPE-2k  vocab size:", bpe_2k.get_vocab_size())
print("BPE-10k vocab size:", bpe_10k.get_vocab_size())

bpe_2k.save(f"{OUT_DIR}/bpe_2k.json")
bpe_10k.save(f"{OUT_DIR}/bpe_10k.json")